# PAN 2 — Modèle du monde et raisonnement simulatif (GLP)

PAN 2 (Institute of Foundation Models — IFM, MBZUAI) est un **modèle du monde
généraliste** : il simule comment le monde change en réponse à des actions. Entrée = une
observation (image optionnelle et texte) et des actions décrites en texte ; sortie = la
**vidéo de l'issue**, jusqu'à 20 s. Son cas d'usage est le **raisonnement simulatif** : un
agent imagine les issues de ses actions candidates *avant* de choisir.

Ce carnet porte la **technique**, pas les poids. Le modèle PAN 2 est un *paramètre
d'exécution* (backbone MMDiT de 29 B paramètres, servi sur 4×B300 / 8×H200) ; la
technique, elle, est entièrement rejouable sur CPU : elle est encodée dans un outil de la
bibliothèque standard, `scripts/genai/pan2/world_model_formalism.py`, dont les **sept
contrôles** sont rejoués dans la section 4.

**Sources** (distillation sourcée, `docs/research/pan2-world-model-analyse.md`) :

| Source | Ce qu'on en tire |
|---|---|
| Xing, Deng, Hou — *Critique of World Model* (arXiv 2507.05169, juin 2026) | la thèse, les cinq dimensions de design, l'architecture GLP, l'instanciation PAN |
| Model card `IFM/PAN2` (Hugging Face) | l'architecture concrète, les checkpoints, la recette vLLM |
| Recette vLLM officielle `recipes.vllm.ai/IFM/PAN2` | la commande de service |

Le fil rouge du carnet : **simuler** → **formaliser** → **exercer**.

## Section 1 — La thèse : un modèle du monde est un simulateur de possibilités actionnables

Le papier ouvre sur *Dune* et sur le **Kwisatz Haderach**, qui « simule les issues de
tous les plans possibles pour tracer la meilleure voie ». Sa thèse tient en une phrase
(Abstract, p. 1) : le but premier d'un modèle du monde est de **simuler toutes les
possibilités actionnables du monde réel, pour un raisonnement et une action finalisés**.

Opérationnellement, un modèle du monde (WM) est un modèle génératif qui, à l'état du
monde $s$ et à une action $a$, prédit l'état suivant $s'$ par une **distribution
conditionnelle** (Éq. 1, p. 2) :

$$s' \sim p(s' \mid s, a)$$

Le point de vue porteur traverse tout le papier (p. 2) : un modèle du monde est
**fondamentalement un simulateur, pas un générateur de contenu visuel** — c'est le substrat
du **raisonnement simulatif**, la capacité d'incarner mentalement des scénarios
hypothétiques au service d'une décision finalisée.

La première cellule charge l'outil de la formalisation et affiche cette thèse avec la
factorisation de PAN 2.

In [1]:
"""PAN 2 / GLP — la thèse, et l'outil qui rend la technique rejouable sur CPU.

Ce carnet n'exécute PAS le modèle PAN 2 (29 B paramètres, servi sur 4xB300 / 8xH200,
recette rappelée dans la dernière section) : il exécute la FORMALISATION de la technique,
encodée dans un outil de la bibliothèque standard, sans dépendance et sans GPU.
"""
from __future__ import annotations

import importlib.util
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for candidat in (REPO_ROOT, *REPO_ROOT.parents):
    if (candidat / "scripts" / "genai" / "pan2" / "world_model_formalism.py").is_file():
        REPO_ROOT = candidat
        break

TOOL_PATH = REPO_ROOT / "scripts" / "genai" / "pan2" / "world_model_formalism.py"
print("python    :", platform.python_version())
# On n'affiche que le chemin de l'outil RELATIF a la racine : un chemin absolu
# de poste ne doit pas se retrouver dans une sortie commitee.
print("outil     :", TOOL_PATH.relative_to(REPO_ROOT).as_posix() if TOOL_PATH.is_file() else "INTROUVABLE")

if not TOOL_PATH.is_file():
    raise FileNotFoundError(f"outil de la formalisation introuvable : {TOOL_PATH}")

spec = importlib.util.spec_from_file_location("world_model_formalism", TOOL_PATH)
formalism = importlib.util.module_from_spec(spec)
sys.modules["world_model_formalism"] = formalism
spec.loader.exec_module(formalism)

print()
print("These (arXiv 2507.05169, p. 1) :")
print("  le but premier d'un modele du monde est de SIMULER toutes les possibilites")
print("  actionnables du monde reel, pour un raisonnement et une action finalises.")
print()
print("Modele du monde (Eq. 1, p. 2) : s' ~ p(s' | s, a)")
print("  s : etat courant | a : action candidate | s' : etat simule suivant")
print()
print("Factorisation PAN 2 (p. 22-23) : p_PAN(o' | o, a) =", " . ".join(formalism.GLP.factors()))

python    : 3.13.16
outil     : scripts/genai/pan2/world_model_formalism.py

These (arXiv 2507.05169, p. 1) :
  le but premier d'un modele du monde est de SIMULER toutes les possibilites
  actionnables du monde reel, pour un raisonnement et une action finalises.

Modele du monde (Eq. 1, p. 2) : s' ~ p(s' | s, a)
  s : etat courant | a : action candidate | s' : etat simule suivant

Factorisation PAN 2 (p. 22-23) : p_PAN(o' | o, a) = p_h(s_hat|o) . p_f(s_hat'|s_hat,a) . p_g(o'|s_hat')


### Lecture 1 — trois choses que la cellule établit

**Un modèle du monde ne prédit pas une image, il prédit un état.** L'équation
`s' ~ p(s' | s, a)` ne parle pas de pixels : elle parle d'un **état** du monde et d'une
**action** qui le transforme. Ce glissement est la thèse du papier : le simulateur est
première, l'image n'est qu'une observation parmi d'autres — s'y ajoutent le son, la
température, le mouvement, les signaux incarnés et le texte, que le décodeur de PAN
reconstruit ensemble (« l'expérience sensorielle complète », §4 de l'analyse).

**La factorisation est en trois morceaux, pas en un.** `p_PAN(o' | o, a)` se décompose en
un encodeur `p_h`, un modèle du monde `p_f` et un décodeur `p_g`. Cette décomposition
n'est pas cosmétique : c'est elle qui rendra discutable la variante à deux morceaux de la
section 3, et c'est elle qui permet de situer *où* une information peut se perdre.

**L'outil est de la bibliothèque standard, sans GPU.** C'est ce qui permet à un carnet de
la série Video de porter la technique sans dépendre d'un GPU à 8×H200 : la formalisation
est encodée, donc vérifiable, donc rejouable par le lecteur. Les poids du modèle ne sont
pas dans le dépôt — ils restent un paramètre d'exécution.

## Section 2 — Les cinq dimensions de design, et la position GLP

Le papier n'examine pas les modèles du monde un par un : il les range sur **cinq
dimensions de design** (§4, p. 2–3), et y situe la position de GLP. Les cinq dimensions
sont :

| Dimension | Question posée | Position GLP / PAN |
|---|---|---|
| **Data** | quelles données portent l'information voulue du monde ? | toutes les modalités d'expérience ; **densité d'information**, pas seulement volume (§4.1) |
| **Representation** | quel espace latent pour l'état du monde ? | **mixte continu + discret**, hiérarchique, à plusieurs niveaux (§4.2) |
| **Architecture** | comment raisonner sur les représentations ? | **GLP** : bottleneck génératif + backbone LLM + prédicteur de diffusion (§4.3) |
| **Objective** | quel objectif guide l'entraînement ? | perte générative **ancrée dans les données d'observation** (§4.4) |
| **Usage** | comment le WM entre-t-il dans un système décisionnel ? | simuler l'expérience pour **entraîner des agents par RL** (§4.5) |

La même formalisation encode aussi les **trois niveaux de prédiction latente** du GLP
(§4.3.3, p. 14–15), que la cellule affiche avec les dimensions.

In [2]:
# Les dimensions et les niveaux sont lus dans la bibliotheque du formalisme :
# ce carnet n'en redéfinit aucun, il affiche ce que le module formalise.
print("Cinq dimensions de design (papier §4, p. 2-3) et position GLP :\n")
for dimension in formalism.DIMENSIONS:
    print(f"  - {dimension:15s} {formalism.GLP_POSITION[dimension]}")

print("\nTrois niveaux de prediction latente du GLP (§4.3.3, p. 14-15) :\n")
for niveau in formalism.GLP_LEVELS:
    print(f"  - niveau {niveau.name:14s} {niveau.mechanism}")
    print(f"    gere : {niveau.handles}")

# La dimension 'usage' ferme le cadre : la simulation sert a l'entrainement
# des agents par reinforcement learning, pas a produire de la video.
print("\nLa dimension 'usage' ferme le cadre : la simulation sert a entrainement")
print("des agents par reinforcement learning, pas a produire de la video.")

Cinq dimensions de design (papier §4, p. 2-3) et position GLP :

  - data            toutes les modalités d'expérience; densité d'information, pas seulement le volume
  - representation  mixte continu + discret, hiérarchique, multi-niveaux
  - architecture    GLP: bottleneck génératif + backbone LLM + prédicteur de next-embedding par diffusion
  - objective       perte générative ancrée dans les données d'observation (espace données, pas seulement latent)
  - usage           simuler l'expérience pour entraîner des agents par RL

Trois niveaux de prediction latente du GLP (§4.3.3, p. 14-15) :

  - niveau bas            next-embedding (diffusion latente)
    gere : perceptuel continu fin-grain: pixels, audio, proprioception
  - niveau intermédiaire  next-token (Transformer autoregressif, tokens type VQ-VAE)
    gere : structure symbolique et compositionnelle
  - niveau haut           LLM en 'thought space' (tokens langue)
    gere : planification long-horizon, simulation mentale, contre-

### Lecture 2 — cinq dimensions, une seule décision à chaque fois

**Chaque dimension est une question, pas un adjectif.** « Data » ne veut pas dire
« beaucoup de données » : le papier précise **densité d'information**, pas volume
(§4.1). C'est une distinction que le lecteur application garde en tête : un corpus plus
gros mais moins dense ne change rien à ce que le modèle peut décider.

**La position GLP est une réponse à chaque question, pas une déclaration générale.**
Sur *architecture*, GLP répond « bottleneck génératif + backbone LLM + prédicteur de
diffusion » ; sur *objective*, il répond « perte générative ancrée dans les données
d'observation », c'est-à-dire **l'espace des données** et pas seulement l'espace latent.
Cette nuance porte tout le reproche de la section 3 : une perte qui ne s'ancre que dans
l'espace latent laisse l'encodeur libre de jeter ce qui compte.

**Les trois niveaux répondent à la même question par trois granularités.** Le niveau
**bas** (next-embedding par diffusion) porte le stochastique et le fine-grain perceptuel ;
le niveau **intermédiaire** (next-token sur tokens modaux discrets) porte la structure
symbolique ; le niveau **haut** (LLM en « thought space ») porte la planification
long-horizon et le raisonnement contre-factuel. Le mot « GLP » ne désigne pas une
architecture unique mais une **hiérarchie de prédictions**.

**Le carnet ne perd pas de vue l'usage final.** La dernière ligne de la cellule le
rappelle : l'agent simule pour **apprendre**, pas pour regarder une vidéo. C'est ce qui
distingue un modèle du monde d'un générateur vidéo, et c'est ce que la section 6 met en
scène.

## Section 3 — Deux boucles : l'encodeur-encodeur (JEPA) et l'encodeur-décodeur (GLP)

Deux approches se disputent le terrain du modèle du monde (§4.3.1) :

- l'**encodeur-décodeur** (Genie 2, NVIDIA Cosmos) privilégie la **validation** : il prédit
  directement l'observation suivante $o'$ à partir de $o$ et $a$, en **boucle fermée** — la
  prédiction est confrontée à ce qui est observable ;
- **JEPA** privilégie l'**abstraction** : design encodeur-encodeur, non autoregressif et
  non génératif, qui prédit l'état latent suivant directement, en **boucle ouverte** — sans
  reconstruire l'observation.

Le reproche du papier porte sur un point précis (§3.2 de l'analyse, p. 12–13). JEPA définit
$\hat s = h(o)$ et $\hat s' = f(\hat s, a)$ ; son application récursive est
**fonctionnellement autoregressive et générative**, bien qu'aucun décodeur probabiliste
explicite n'existe. Et sa distribution conditionnelle est **dégénérée** :

$$p_f(\hat s' \mid \hat s, a) = \delta(\hat s' - f(\hat s, a))$$

où $\delta(\cdot)$ est une **masse de Dirac** centrée sur la prédiction déterministe.
JEPA n'est donc pas génératif *au sens probabiliste* — il ne modélise pas l'incertitude.

Le défaut structurel est ailleurs : l'encodeur $h$ de JEPA est **délibérément perdant et
non inversible**. S'il jette l'information qui compte pour la dynamique prédite, alors
« aucune sophistication du prédicteur ne peut récupérer ce qui a été jeté » (p. 13).

C'est là que la **boucle fermée générative** apporte un mécanisme de diagnostic intégré :
*si le décodeur ne peut pas reconstruire l'observation suivante à partir de l'état prédit,
l'encodeur était trop perdant*.

In [3]:
# Les deux boucles sont lues dans le formalisme : JEPA sans decodeur (boucle
# ouverte), GLP avec decodeur (boucle fermee).
print("Deux boucles, deux factorisations :\n")
for nom, modele in (("JEPA", formalism.JEPA), ("GLP", formalism.GLP)):
    etat = "boucle FERMEE (decodeur present)" if modele.is_closed_loop() else "boucle OUVERTE (decodeur absent)"
    print(f"  {nom:5s} : {etat}")
    for facteur in modele.factors():
        print(f"          {facteur}")
    print()

# La difference de NATURE se lit dans la distribution de l'etat suivant : une
# masse de Dirac (un seul point) contre une loi generative (support etendu).
print("Distribution de l'etat suivant :\n")
jepa = formalism.DiracDelta(center=0.0)
glp = formalism.GenerativeDistribution()
print(f"  JEPA  (masse de Dirac)   : support = {jepa.support_size()} point  | probabiliste = {jepa.is_probabilistic()}")
print(f"  GLP   (generative propre): support = {glp.support_size()} points | probabiliste = {glp.is_probabilistic()}")

# Schema comparatif : illustration de la propriete formelle, PAS une mesure
# du modele (aucun poids du GLP n'est charge ici).
print("\nSchema comparatif (illustration de la propriete formelle, PAS une mesure du modele) :")
print("  JEPA : p_f(s'|s,a) = delta(s' - f(s,a))  ->  une seule realite possible")
print("           0.0 | #")
print("  GLP  : loi generative, support etendu   ->  plusieurs realites possibles")
for valeur in (0.0, 1.0):
    print(f"          {valeur:>4} | #")

Deux boucles, deux factorisations :

  JEPA  : boucle OUVERTE (decodeur absent)
          p_h(s_hat|o)
          p_f(s_hat'|s_hat,a)

  GLP   : boucle FERMEE (decodeur present)
          p_h(s_hat|o)
          p_f(s_hat'|s_hat,a)
          p_g(o'|s_hat')

Distribution de l'etat suivant :

  JEPA  (masse de Dirac)   : support = 1 point  | probabiliste = False
  GLP   (generative propre): support = 2 points | probabiliste = True

Schema comparatif (illustration de la propriete formelle, PAS une mesure du modele) :
  JEPA : p_f(s'|s,a) = delta(s' - f(s,a))  ->  une seule realite possible
           0.0 | #
  GLP  : loi generative, support etendu   ->  plusieurs realites possibles
           0.0 | #
           1.0 | #


### Lecture 3 — le décodeur, pièce à conviction

**Le décodeur est un diagnostic, pas un gadget de rendu.** C'est la thèse la plus
contre-intuitive de l'article, et la cellule la met en forme : `is_closed_loop()` est
`True` pour GLP et `False` pour JEPA, et cette seule différence de présence du facteur
`p_g(o'|s_hat')` change la nature du modèle. Chez JEPA, la perte peut être minimisée
dans l'espace latent sans que rien n'oblige l'espace latent à rester informatif de la
réalité ; chez GLP, la reconstruction de l'observation **impose** que l'état prédit
contienne l'information pertinemment dynamique. Le décodeur maintient ainsi « en
permanence la pression pour que la représentation conserve toute l'information
dynamiquement pertinente » (p. 13).

**Dégénéré ne veut pas dire faux.** La masse de Dirac est une distribution
*concentrée*, pas une absence de distribution : `support_size() == 1` signifie qu'il n'y
a **qu'une seule réalité possible** pour la prédiction donnée, pas qu'aucune prédiction
n'est faite. Le schéma comparatif le montre : JEPA dessine une barre, GLP en dessine
deux. C'est exactement la distinction *probabiliste* / *fonctionnelle* du papier — JEPA
n'est pas génératif au sens probabiliste, mais il simule récursivement l'évolution
latente, et hérite donc des mêmes problèmes que les modèles autoregressifs.

**Le reproche est un argument d'information, pas de goût.** « Aucune sophistication du
prédicteur ne peut récupérer ce qui a été jeté » : un encodeur perdant non inversible ne
peut pas produire une représentation *stateful*, c'est-à-dire une statistique suffisante
de la dynamique. Ajoute-t-on un meilleur prédicteur ? L'information n'est plus là.

**Ce que la boucle fermée coûte.** Elle ne se contente pas de valider : elle rend la
perception **nécessaire** à la boucle. Un modèle du monde qui ne rejoue jamais son
observation ne peut pas être corrigé par l'observation — il peut être *agrégé* par elle,
ce qui est une autre chose, et ce que la section 5 héritera de GLP.

## Section 4 — La factorisation PAN 2 et ses sept contrôles

PAN (Physical, Agentic, Nested) instancie le GLP dans un monde à **représentation
mixte** et à **raisonnement multi-échelle** (§4 de l'analyse) :

- **Encodeur sensoriel $h$** — voie **dual** : un *tokenizer* mappe hiérarchiquement le
  signal brut en **tokens discrets** (tokens abstraits type VQ-VAE et mots de la langue
  naturelle), à nombre flexible et **stateful par construction** ; les détails bas niveau
  sont gardés en **embeddings latents continus**. L'état du monde est donc en couches
  $\hat s = \{\hat s_i\}_{i=1}^{N}$ ;
- **Backbone $f$** — un **LLM enrichi** (raisonnement discret conceptuel) *et* un
  **prédicteur de next-latent-embedding par diffusion** (raisonnement perceptuel continu) ;
  un *Learned Switch* combine les deux selon la demande de la tâche ;
- **Décodeur multimodal $g$** — reconstruit l'observation suivante $\hat o'$ — pas seulement
  la vidéo, mais l'**expérience sensorielle complète** — et la compare à $o'$.

La factorisation formelle (p. 22–23) :

$$p_{PAN}(o' \mid o, a) = \sum_{\hat s, \hat s'} \underbrace{p_h(\hat s \mid o)}_{\text{encodeur}} \;\underbrace{p_f(\hat s' \mid \hat s, a)}_{\text{modèle du monde}} \;\underbrace{p_g(o' \mid \hat s')}_{\text{décodeur}}$$

C'est ce décodeur $g$, absent de JEPA, qui ferme la boucle et distingue le GLP d'un JEPA.
L'outil encode ces faits et les soumet à **sept contrôles** — la cellule les rejoue.

In [4]:
code = formalism.run_checks()
print()
print(f"Code de sortie de run_checks() : {code}  (0 = tous les controles PASS)")
assert code == 0, "un controle a echoue : la formalisation n'est plus celle du papier"

world_model_formalism — 7 contrôles (GLP / PAN 2, arXiv 2507.05169)
[PASS] dimensions         5 dimensions de design, GLP positionné sur chacune (manquants: aucun)
[PASS] glp-niveaux        3 niveaux du GLP = ['bas', 'intermédiaire', 'haut']
[PASS] jepa-degenerée     JEPA = masse de Dirac (support ponctuel, non probabiliste) -> dégénérée
[PASS] glp-generative     GLP = distribution générative propre (support etendu, probabiliste)
[PASS] boucle-fermée      le décodeur g ABSENT de JEPA, PRESENT de GLP (ferme la boucle)
[PASS] factorisation-PAN  p_PAN(o'|o,a) = p_h . p_f . p_g (3 facteurs)
[PASS] propriété-diagnostic boucle fermée (décodeur) <=> supervision ancrée dans l'observation (diagnostic)
--- 7/7 PASS, 0 FAIL ---

Code de sortie de run_checks() : 0  (0 = tous les controles PASS)


### Lecture 4 — sept contrôles, et ce qu'ils prouvent

**Un contrôle qui passe ne prouve pas que le modèle marche.** Il prouve que la
*formulation* tient : que les sept affirmations qu'on peut dériver du papier sont
mutuellement cohérentes et correspondent à ce qu'on a codé. C'est la différence entre
« le carnet de la technique » et « le modèle » : PAN 2 est un modèle de 29 milliards de
paramètres, servi sur 4×B300 ou 8×H200 — il n'est **pas** dans ce carnet, et aucun
résultat de ce carnet ne constitue une mesure de PAN 2.

**Les sept contrôles se lisent comme les sept thèses du papier.**

| Contrôle | Thèse vérifiée |
|---|---|
| `dimensions` | il y a bien **cinq** dimensions de design, et GLP est positionné sur chacune |
| `glp-niveaux` | les trois niveaux sont *bas*, *intermédiaire*, *haut* — dans cet ordre |
| `jepa-degenerée` | la distribution de JEPA est **dégénérée** : support ponctuel, non probabiliste |
| `glp-generative` | celle de GLP est **propre** : support étendu, probabiliste |
| `boucle-fermée` | le décodeur est **absent** de JEPA, **présent** de GLP |
| `factorisation-PAN` | la factorisation PAN comporte **les trois** facteurs $p_h, p_f, p_g$ |
| `propriété-diagnostic` | boucle fermée ⟺ supervision ancrée dans l'observation |

**L'ordre des contrôles suit l'argument du papier**, pas l'ordre du code : on part des
catégories (dimensions, niveaux), on compare les deux distributions, puis on vérifie la
propriété structurelle (le décodeur), et on finit par la formule qui les encapsule.

**Un contrôle qui lève est un FAIL, pas un crash.** `run_checks()` attrape l'exception et
la rend visible comme un FAIL — un organe qui avale une erreur et rend « tout va bien »
serre un contrôle qui ne contrôle plus rien. L'`assert` de la cellule est l'invariant
**positif** : il ne peut échouer que si la formalisation a divergé de l'article.

**Ce que l'exercice ne couvre pas** : la question de savoir si un *modèle* entraîné par
cette factorisation est meilleur qu'un autre. Aucune cellule de ce carnet ne compare
PAN 2 à un JEPA de même taille — ce serait une mesure, et une mesure demande la machine
(les 29 B), pas un carnet CPU.

## Section 5 — La description structurée : ce que l'agent lit

Tout le raisonnement du monde tient dans une **interface** : ce que l'agent sait de l'état
courant, et ce qu'il peut décider. Le GLP rend cette interface **explicite et
hiérarchique** (la position GLP sur la dimension *representation* : « mixte continu +
discret, hiérarchique, multi-niveaux »).

Chez PAN, l'encodeur sensoriel $h$ a **deux voies** :

- une voie **discrète** — un *tokenizer* mappe le signal brut en **tokens** (tokens
  abstraits de type VQ-VAE, et les mots de la langue naturelle) : c'est le canal
  **symbolique**, compositionnel, hiérarchique ;
- une voie **continue** — les détails bas niveau sont conservés en **embeddings latents** :
  c'est le canal **perceptuel**, fin, non discrétisé.

L'état du monde est donc **en couches** $\hat s = \{\hat s_i\}_{i=1}^{N}$, et non un vecteur
unique. Cette structure est ce qui rend l'état *stateful* : les couches portent des
granularités différentes, et une information utile à l'échelle fine (un pixel, une
proprioception) ne disparaît pas quand une abstraction est extraite.

**Pourquoi c'est la pièce qui manque à JEPA.** L'encodeur de JEPA est **délibérément
perdant** : on lui demande de compresser, et ce qui est compressé est perdu. La
description structurée de GLP est l'inverse d'un choix : garder les deux étages — le
discret pour la structure, le continu pour le détail — et **faire porter la preuve par le
décodeur** que l'ensemble reste informatif.

La cellule suivante pose un **schéma didactique** de cette description — il illustre la
forme (une description qui porte des événements, des variables, et se valide), il
n'est **pas** une sortie de PAN 2 et ne prétend pas reproduire l'encodeur réel.

In [5]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Evenement:
    """Un evenement observable : un instant, un type, un contenu."""

    t: float
    kind: str
    content: str


@dataclass
class DescriptionEtat:
    """La description structuree d'un etat.

    SCHEMA DIDACTIQUE : il illustre la FORME d'une description (des evenements,
    des variables, une validation, un rendu textuel que l'agent lit). Ce n'est
    pas une sortie de PAN 2, et il ne reproduit pas l'encodeur sensoriel h.
    """

    evenements: list[Evenement] = field(default_factory=list)
    variables: dict[str, float] = field(default_factory=dict)

    def ajouter(self, evenement: Evenement) -> None:
        self.evenements.append(evenement)

    def valider(self) -> bool:
        """Le temps est monotone et les variables sont finies."""
        temps = [e.t for e in self.evenements]
        if any(b < a for a, b in zip(temps, temps[1:])):
            return False
        return all(v == v and abs(v) != float("inf") for v in self.variables.values())

    def rendre(self) -> str:
        """La description en texte : c'est elle que l'agent lit et planifie."""
        instant = self.evenements[-1].t if self.evenements else 0.0
        lignes = [f"etat a t={instant:.1f}"]
        for e in self.evenements[-3:]:
            lignes.append(f"  - t={e.t:.1f} {e.kind}: {e.content}")
        for nom, valeur in sorted(self.variables.items()):
            lignes.append(f"  = {nom} = {valeur:.3f}")
        return "\n".join(lignes)


etat = DescriptionEtat(variables={"vitesse": 2.0, "distance": 12.5})
etat.ajouter(Evenement(t=0.0, kind="action", content="avancer"))
etat.ajouter(Evenement(t=1.0, kind="observation", content="la porte s'ouvre"))
print(etat.rendre())
print("description valide :", etat.valider())

# Contre-exemple : un pas de temps retrograde invalide la description.
etat_bidon = DescriptionEtat(variables={"vitesse": 1.0})
etat_bidon.ajouter(Evenement(t=2.0, kind="observation", content="b"))
etat_bidon.ajouter(Evenement(t=1.0, kind="observation", content="a"))
print("description retrograde valide :", etat_bidon.valider())

etat a t=1.0
  - t=0.0 action: avancer
  - t=1.0 observation: la porte s'ouvre
  = distance = 12.500
  = vitesse = 2.000
description valide : True
description retrograde valide : False


### Lecture 5 — une description se valide, ou n'est pas une description

**La forme est l'essentiel.** Une description d'état n'est pas un texte libre : c'est une
structure qui se **valide**. Le schéma pose deux contrôles qui suffisent à rendre la
description utilisable par un planificateur :

- le **temps est monotone** — les événements sont ordonnés, et une description dont le
  temps recule est une description *incohérente*, pas une description *pauvre* ;
- les **variables sont finies** — pas de `NaN`, pas d'infini ; ce sont les valeurs dont on
  calcule une récompense, et une récompense infinie est une récompense qui ne dit rien.

Le contre-exemple de la cellule est là pour cela : une description dont les événements
remontent le temps **échoue** à la validation. Ce n'est pas une garde défensive
inutilisée — c'est le refus d'un état que le modèle du monde ne pourrait pas propager
sans se contredire.

**Rendre l'état lisible est un acte de conception, pas un confort.** `rendre()` produit un
texte que l'agent *lit* et sur lequel il *planifie* : le passage d'une structure à un
texte n'est pas une mécanique, c'est l'interface entre le latent et le raisonnement
discret. Dans GLP, ce passage est porté par le **niveau haut** — le LLM en « thought
space », dont les tokens de langue sont justement ce format textuel.

**Ce que le schéma ne fait pas, et pourquoi c'est honnête.** Il ne compresse rien, il ne
prédit rien, il n'est pas entraîné. Il montre la *forme* qu'une description doit prendre
pour être planifiable — et il le déclare dans sa docstring. Un schéma qui se prend pour
l'encodeur de PAN serait un faux ; un schéma qui montre la forme, et le dit, est un
outil pédagogique.

## Section 6 — Le raisonnement simulé : comparer des plans sans les vivre

La position GLP sur la dimension *usage* est explicite : « simuler l'expérience pour
entraîner des agents par RL ». Le modèle du monde n'est donc pas un prédicteur décoratif :
c'est un **milieu** où l'on essaie des plans.

Le mécanisme, en GLP, est la **généralisation latente** : le prédicteur de
*next-embedding* par diffusion opère dans l'espace latent, ce qui permet de **mapper**
une trajectoire latente **sans** décoder chaque image intermédiaire. Le gain est
énorme — on teste dix mille trajectoires latentes au lieu de dix mille vidéos. Mais
attention : la génération latente seule ne suffit pas pour l'agent ; il faut que la
décision soit **ancrée dans l'observation**, et c'est le décodeur qui fournit cet
ancrage.

La cellule ci-dessous pose un **schéma didactique** de comparaison de plans : trois
séquences d'actions, une simulation déterministe, une récompense inventée pour
l'exercice. Ce n'est pas une simulation de PAN 2 — c'est la **forme** de l'exercice
« comparer deux plans par simulation », avec le contre-exemple qui rend la comparaison
intéressante.

In [6]:
# Schema DIDACTIQUE du carnet : une cible, une transition deterministe, une
# recompense fabriquee pour l'exercice, et trois plans a comparer.
CIBLE = 10


def etat_suivant(etat: int, action: str) -> int:
    """Une transition deterministe. SCHEMA DIDACTIQUE, pas un modele du monde."""
    if action == "avancer":
        return etat + 1
    if action == "tourner":
        return etat - 1
    return etat


def recompense(etat: int) -> float:
    """Recompense DIDACTIQUE : closeness a la cible. Inventee pour l'exercice."""
    return float(CIBLE - abs(etat - CIBLE))


# Trois plans de longueurs differentes : un court, un long, et un piege qui
# s'ecarte de la cible.
plans = {
    "plan court (tourner, avancer)": ["tourner", "avancer"],
    "plan long (avancer x4)": ["avancer"] * 4,
    "plan piege (tourner x2)": ["tourner", "tourner"],
}

etat_initial = 3
print(f"etat initial = {etat_initial}, cible = {CIBLE}\n")

# Pour chaque plan on rejoue les actions, on cumule la recompense et on garde
# la trace : c'est le chemin qui enseigne, pas seulement l'etat final.
for nom, plan in plans.items():
    etat = etat_initial
    total = 0.0
    trace = []
    for action in plan:
        etat = etat_suivant(etat, action)
        total += recompense(etat)
        trace.append(f"{action}->{etat}")
    ecart = abs(etat - CIBLE)
    print(f"{nom:32s} final {etat:3d} | trace {', '.join(trace):28s} | "
          f"recompense {total:6.2f} | ecart {ecart}")

etat initial = 3, cible = 10

plan court (tourner, avancer)    final   3 | trace tourner->2, avancer->3       | recompense   5.00 | ecart 7
plan long (avancer x4)           final   7 | trace avancer->4, avancer->5, avancer->6, avancer->7 | recompense  22.00 | ecart 3
plan piege (tourner x2)          final   1 | trace tourner->2, tourner->1       | recompense   3.00 | ecart 9


### Lecture 6 — comparer deux plans, c'est déjà raisonner

**La simulation ne remplace pas l'expérience : elle la précède.** Les trois plans de la
cellule n'aboutissent pas au même état : le plan long finit à 7 (écart 3), le plan court
revient où il était parti (écart 7), le plan piège s'éloigne (écart 9). Ce que la cellule
met en scène, c'est le geste central du RL sur modèle du monde : **évaluer une
trajectoire avant de la subir**, dans un environnement qui coûte un calcul et non une
victoire.

**Le total cumulé est un artefact ; l'écart final est la mesure.** La récompense cumulée
favorise le plan long (22.00 contre 5.00 et 3.00) — mais le plan long cumule **quatre
termes**, le plan court **deux**. Cumuler une récompense par pas favorise mécaniquement
le plan le plus long : le total n'est pas comparable entre plans de longueurs différentes.
La grandeur qui se compare, c'est l'**écart à la cible** — et sur celle-ci le classement
est net : 3 pour le plan long, 7 pour le court, 9 pour le piège.

**La trace est plus instructive que les deux chiffres.** La suite des états dit *pourquoi* :
le plan court a « tourné puis avancé » et se retrouve exactement à son point de départ ; le
plan piège s'est éloigné en tournant deux fois. Un plan qui converge et un plan qui
dérive ne se distinguent pas par un total, ils se distinguent par une **trace** — et c'est
pourquoi le décodeur de GLP reconstruit la trajectoire : sans reconstruction, on n'a
qu'un score, pas une explication.

**Ce que la simulation de la cellule ne fait pas.** La transition est déterministe et la
récompense est une distance à la cible : il n'y a **aucune incertitude**, donc aucune
distribution. Or c'est précisément l'incertitude que GLP modélise et que JEPA ne modélise
pas. Un schéma qui figerait l'incertitude demanderait le *prédicteur de next-embedding
par diffusion* — c'est-à-dire, pour de vrai, le modèle. La cellule déclare donc son
statut : elle illustre la **forme** de la comparaison, elle ne la mesure pas.

## Exercices

Trois exercices, du schéma à la formule. Les énoncés sont sous chaque stub ; le code
est à compléter dans les cellules de code qui suivent.

### Exercice 1 — décrire un état

Écrivez `decrire_etat(evenements, variables)` qui renvoie une `DescriptionEtat`
 contenant les événements (dans l'ordre reçu) et les variables.

**Indice :** `DescriptionEtat(variables=variables)` puis `ajouter()` chaque événement.

In [7]:
def decrire_etat(evenements, variables):
    """TODO etudiant : construire la DescriptionEtat a partir des evenements et des variables.

    Indice : DescriptionEtat(variables=variables), puis ajouter() chaque evenement,
    puis renvoyer l'objet construit.
    """
    resultat = None  # TODO etudiant
    return resultat


# Pattern d'appel (decommentez apres avoir complete la fonction) :
# etat = decrire_etat([Evenement(t=0.0, kind="action", content="avancer")], {"vitesse": 2.0})
# print(etat.rendre())
print("Exercice 1 - decrire_etat a completer (stub).")

Exercice 1 - decrire_etat a completer (stub).


### Exercice 2 — vérifier une description

Écrivez `valider_description(evenements, variables)` qui renvoie `True` si la description
est valide : temps monotone, variables finies.

**Indice :** construisez d'abord la description (exercice 1), puis renvoyez
`description.valider()`.

In [8]:
def valider_description(evenements, variables):
    """TODO etudiant : renvoyer True si la description est valide.

    Indice : construire la description avec decrire_etat(...), puis renvoyer
    description.valider().
    """
    resultat = None  # TODO etudiant
    return resultat


# Pattern d'appel (decommentez apres avoir complete la fonction) :
# print(valider_description([Evenement(t=0.0, kind="action", content="avancer")], {"vitesse": 2.0}))
print("Exercice 2 - valider_description a completer (stub).")

Exercice 2 - valider_description a completer (stub).


### Exercice 3 — factoriser

Écrivez `factoriser_pan()` qui renvoie la liste des trois facteurs de la factorisation
$p_{PAN}(o' \mid o, a) = p_h \cdot p_f \cdot p_g$, dans l'ordre.

**Indice :** le module `formalism` expose `GLP.factors()`.

In [9]:
def factoriser_pan():
    """TODO etudiant : renvoyer la liste des trois facteurs de p_PAN(o'|o,a).

    Indice : formalism.GLP.factors() renvoie deja la liste, dans l'ordre
    p_h, p_f, p_g.
    """
    resultat = None  # TODO etudiant
    return resultat


# Pattern d'appel (decommentez apres avoir complete la fonction) :
# print(factoriser_pan())
print("Exercice 3 - factoriser_pan a completer (stub).")

Exercice 3 - factoriser_pan a completer (stub).


## À retenir

- **Un modèle du monde est un simulateur**, pas un prédicteur de vidéo : sa valeur est
  dans ce qu'il permet d'*essayer* avant de le vivre.
- **Cinq dimensions de design** : *data*, *representation*, *architecture*, *objective*,
  *usage*. GLP est positionné sur chacune — et une dimension non décidée est une
  architecture non décidée.
- **JEPA est dégénéré** : sa loi sur l'état suivant est une masse de Dirac (support
  ponctuel, non probabiliste). Il simule la fonction, pas la loi.
- **Le décodeur est un diagnostic** : c'est lui qui ancre la représentation dans
  l'observation, et donc qui révèle une représentation trop perdante. Sans lui, aucune
  dérive ne remonte.
- **PAN factorise en trois** : $p_{PAN}(o' \mid o, a) = p_h \cdot p_f \cdot p_g$ —
  encodeur, modèle du monde, décodeur. Le troisième facteur est **ce qui distingue GLP**.
- **Trois niveaux de prédiction** : *bas* (next-embedding), *intermédiaire* (next-token),
  *haut* (thought space du LLM).
- **Un carnet de technique ne mesure pas le modèle** : les sept contrôles vérifient la
  *formulation*, pas les 29 milliards de paramètres.

## Pour aller plus loin

- **L'article** — Xing, Deng, Hou, *Critique of World Model*, arXiv 2507.05169 : la thèse,
  les cinq dimensions (§4), la critique de JEPA (§3.2), la factorisation (§4.3, p. 22-23).
- **La model card** `IFM/PAN2` (Hugging Face) et la **recette vLLM** officielle :
  l'infrastructure de service du modèle (29 B, architecture MMDiT ; 4×B300 ou 8×H200).
  Ce carnet reste CPU : le servir demande la flotte GPU, pas ce poste.
- **L'analyse interne du dépôt** : `docs/research/pan2-world-model-analyse.md` — la
  distillation dont ce carnet est la mise en forme exécutable.
- **L'outil** : `scripts/genai/pan2/world_model_formalism.py` — la formalisation
  vérifiable, CPU, sans dépendance (`--check` pour les sept contrôles).
- **Le gymnase** : `MyIA.AI.Notebooks/GenAI/Video/02-Advanced/` — les carnets voisins
  de la même série (MiniMax-H3, Wan, Hunyuan) pour la comparaison d'architecture.

**Navigation serie Video** : [02-1 HunyuanVideo](02-1-HunyuanVideo-Generation.ipynb) · [02-2 LTX-Video](02-2-Generation-Video-Legere.ipynb) · [02-3 Wan](02-3-Wan-Video-Generation.ipynb) · [02-4 SVD](02-4-SVD-Image-to-Video.ipynb) · [02-5 LTX-2](02-5-Video-Audio-Conjoint.ipynb) · [02-6 MiniMax H3](02-6-Video-Architecture-Licensing.ipynb) · [02-7 CogVideoX](02-7-CogVideoX-Text-to-Video.ipynb) · **02-8 PAN 2 / GLP (ce notebook)**
